In [1]:
!pip install pymanopt autograd

In [ ]:
import time
import urllib.request
import urllib.error
import numpy as np
import networkx as nx
import cvxpy as cp

# Riemannian Optimization Libraries
try:
    import autograd.numpy as anp
    import pymanopt
    from pymanopt.manifolds import Oblique
    from pymanopt.optimizers import TrustRegions
    PYMANOPT_AVAILABLE = True
except ImportError:
    print("WARNING: 'pymanopt' or 'autograd' not installed. Burer-Monteiro route will fail.")
    PYMANOPT_AVAILABLE = False

# =========================
# 1. Online Benchmark Fetcher
# =========================
def fetch_gset_graph(gset_id="G1"):
    url = f"https://web.stanford.edu/~yyye/yyye/Gset/{gset_id}"
    print(f"Fetching {gset_id} from Stanford repository...")
    
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        response = urllib.request.urlopen(req)
        lines = response.read().decode('utf-8').splitlines()
    except Exception as e:
        print(f"Failed to download {gset_id}: {e}")
        return None

    first_line = lines[0].strip().split()
    n = int(first_line[0])
    
    G = nx.Graph()
    G.add_nodes_from(range(1, n + 1)) 
    
    for line in lines[1:]:
        parts = line.strip().split()
        if len(parts) >= 3:
            u, v, w = int(parts[0]), int(parts[1]), int(parts[2])
            G.add_edge(u, v, weight=w)
            
    G.name = f"GSET_{gset_id}"
    return G

# =========================
# 2. Graph Generator
# =========================
def generate_graphs(configs):
    graphs = []
    
    for idx, config in enumerate(configs):
        g_type = config.get('type')
        n = config.get('n', 50)
        seed = config.get('seed', 42 + idx)
        
        G = None 
        
        if g_type == 'gset':
            # Assuming fetch_gset_graph is a custom function defined elsewhere
            G = fetch_gset_graph(config.get('id', 'G1'))
            if G is not None: 
                graphs.append(G)
            continue
            
        elif g_type == 'erdos_renyi': 
            G = nx.erdos_renyi_graph(n, config.get('p', 0.3), seed=seed)
        elif g_type == 'barabasi_albert': 
            G = nx.barabasi_albert_graph(n, config.get('m', 3), seed=seed)
        elif g_type == 'watts_strogatz': 
            G = nx.watts_strogatz_graph(n, config.get('k', 4), config.get('p', 0.1), seed=seed)
        elif g_type == 'regular': 
            # Note: random_regular_graph takes (d, n)
            G = nx.random_regular_graph(config.get('d', 3), n, seed=seed)
        elif g_type == 'powerlaw_cluster': 
            G = nx.powerlaw_cluster_graph(n, config.get('m', 3), config.get('p', 0.1), seed=seed)
        elif g_type == 'geometric': 
            G = nx.random_geometric_graph(n, config.get('radius', 0.2), seed=seed)
        elif g_type == 'complete': 
            G = nx.complete_graph(n)
        elif g_type == 'complete_bipartite': 
            # Using complete_multipartite_graph as defined in the provided docs
            n1 = config.get('n1', n // 2)
            n2 = config.get('n2', n - n1)
            G = nx.complete_multipartite_graph(n1, n2)
        elif g_type == 'cycle': 
            G = nx.cycle_graph(n)
        elif g_type == 'star': 
            G = nx.star_graph(n - 1)
        elif g_type == 'wheel': 
            G = nx.wheel_graph(n)
        elif g_type == 'circular_ladder': 
            G = nx.circular_ladder_graph(n // 2)
        elif g_type == 'grid_2d': 
            dim1 = config.get('dim1', int(np.sqrt(n)))
            dim2 = config.get('dim2', int(np.sqrt(n)))
            G = nx.grid_2d_graph(dim1, dim2)
        elif g_type == 'hexagonal': 
            # Fixed duplicate 'm' lookup bug from original code
            m_hex = config.get('m', int(np.sqrt(n)))
            n_hex = config.get('n_hex', int(np.sqrt(n)))
            G = nx.hexagonal_lattice_graph(m_hex, n_hex)
        elif g_type == 'hypercube': 
            dim = config.get('dim', max(1, int(np.log2(n) if n > 0 else 3)))
            G = nx.hypercube_graph(dim)
        elif g_type == 'random_tree': 
            # Aligned with the 'Trees' section of the docs
            G = nx.random_labeled_tree(n, seed=seed)
        elif g_type == 'balanced_tree': 
            G = nx.balanced_tree(config.get('r', 2), config.get('h', 3))
        elif g_type == 'karate_club': 
            G = nx.karate_club_graph()
        elif g_type == 'florentine_families': 
            G = nx.florentine_families_graph()
        elif g_type == 'les_miserables': 
            G = nx.les_miserables_graph()
        else:
            print(f"Skipping unknown graph type: '{g_type}'")
            continue

        # Post-processing the generated graph
        if G is not None:
            G = nx.convert_node_labels_to_integers(G)
            G.name = f"{g_type.upper()}_V{G.number_of_nodes()}_E{G.number_of_edges()}_{idx}"
            
            # Ensure every edge has a default weight of 1.0
            for u, v, data in G.edges(data=True):
                if 'weight' not in data:
                    G[u][v]['weight'] = 1.0
                    
            graphs.append(G)
            
    return graphs

# =========================
# 3. Traditional SDP (CVXPY)
# =========================
def solve_sdp_cvxpy(G):
    n = G.number_of_nodes()
    X = cp.Variable((n, n), symmetric=True)
    constraints = [X >> 0, cp.diag(X) == 1]
    L = nx.laplacian_matrix(G, weight='weight').toarray()
    
    prob = cp.Problem(cp.Maximize(cp.trace(L @ X) / 4), constraints)
    prob.solve(solver=cp.SCS)
    
    eigvals, eigvecs = np.linalg.eigh(X.value)
    eigvals = np.clip(eigvals, 0, None)
    V = eigvecs @ np.diag(np.sqrt(eigvals))
    return V, prob.value

# =========================
# 4. Burer-Monteiro SDP
# =========================
def solve_burer_monteiro(G, rank=15):
    n = G.number_of_nodes()
    L = nx.laplacian_matrix(G, weight='weight').toarray()
    manifold = Oblique(rank, n)
    
    @pymanopt.function.autograd(manifold)
    def cost(Y): return -0.25 * anp.trace(Y @ L @ Y.T)

    problem = pymanopt.Problem(manifold, cost)
    optimizer = TrustRegions(verbosity=0)
    Y_opt = optimizer.run(problem).point
    
    V = Y_opt.T
    sdp_val = 0.25 * np.trace(Y_opt @ L @ Y_opt.T)
    return V, sdp_val

# =========================
# 5. Rounding Methods
# =========================

def gw_rounding(G, V, **kwargs):
    """Classic Goemans-Williamson random hyperplane rounding."""
    num_samples = kwargs.get('num_samples', 2000)
    n, d = V.shape
    edges = list(G.edges(data='weight', default=1.0))
    edge_idx = np.array([[u, v] for u, v, w in edges])
    node_to_idx = {node: i for i, node in enumerate(G.nodes())}
    mapped_edges = np.array([[node_to_idx[u], node_to_idx[v]] for u, v in edge_idx])
    weights = np.array([w for u, v, w in edges])

    R = np.random.randn(d, num_samples)
    R /= np.linalg.norm(R, axis=0)
    signs = np.sign(V @ R)
    signs[signs == 0] = 1 

    cut_mask = signs[mapped_edges[:, 0]] != signs[mapped_edges[:, 1]]
    cuts = np.sum(cut_mask * weights[:, np.newaxis], axis=0)
    return cuts[np.argmax(cuts)]

def pca_rounding(G, V, **kwargs):
    """
    Deterministic alternative. Projects the high-dimensional vectors onto 
    their first Principal Component, splitting them by the hyperplane 
    orthogonal to the direction of maximum variance.
    """
    n, d = V.shape
    edges = list(G.edges(data='weight', default=1.0))
    edge_idx = np.array([[u, v] for u, v, w in edges])
    node_to_idx = {node: i for i, node in enumerate(G.nodes())}
    mapped_edges = np.array([[node_to_idx[u], node_to_idx[v]] for u, v in edge_idx])
    weights = np.array([w for u, v, w in edges])

    # Singular Value Decomposition to find the principal component
    _, _, Vh = np.linalg.svd(V, full_matrices=False)
    pc1 = Vh[0, :] # First principal component
    
    # Project and extract signs
    signs = np.sign(V @ pc1)
    signs[signs == 0] = 1
    
    # Calculate cut for this single deterministic partition
    cut_mask = signs[mapped_edges[:, 0]] != signs[mapped_edges[:, 1]]
    return np.sum(cut_mask * weights)



# =========================
# 5.1 Local Search Helpers
# =========================

def _calculate_cut(G, x):
    """Helper: Calculates the total cut value for a partition dictionary x."""
    return sum(data.get('weight', 1.0) for u, v, data in G.edges(data=True) if x[u] != x[v])

def _get_best_gw_partition(G, V, num_samples):
    """Helper: Runs GW rounding and returns the best partition dictionary to use as a warm start."""
    n, d = V.shape
    edges = list(G.edges(data='weight', default=1.0))
    edge_idx = np.array([[u, v] for u, v, w in edges])
    node_to_idx = {node: i for i, node in enumerate(G.nodes())}
    mapped_edges = np.array([[node_to_idx[u], node_to_idx[v]] for u, v in edge_idx])
    weights = np.array([w for u, v, w in edges])

    R = np.random.randn(d, num_samples)
    R /= np.linalg.norm(R, axis=0)
    signs = np.sign(V @ R)
    signs[signs == 0] = 1 

    cut_mask = signs[mapped_edges[:, 0]] != signs[mapped_edges[:, 1]]
    cuts = np.sum(cut_mask * weights[:, np.newaxis], axis=0)
    best_idx = np.argmax(cuts)
    
    best_partition = signs[:, best_idx]
    return {node: best_partition[node_to_idx[node]] for node in G.nodes()}

def greedy_single(G, x):
    """Greedily flips single vertices if it increases the cut."""
    nodes = list(G.nodes())
    improved = True
    
    while improved:
        improved = False
        for u in nodes:
            gain = sum(
                G[u][v].get('weight', 1.0) if x[u] == x[v] else -G[u][v].get('weight', 1.0)
                for v in G.neighbors(u)
            )
            if gain > 0:
                x[u] *= -1
                improved = True
    return x

def greedy_pairwise(G, x):
    """Greedily flips pairs of vertices if it increases the cut."""
    nodes = list(G.nodes())
    improved = True
    
    while improved:
        improved = False
        
        # Precompute individual gains to save massive O(N^2) time
        gains = {}
        for u in nodes:
            gains[u] = sum(
                G[u][v].get('weight', 1.0) if x[u] == x[v] else -G[u][v].get('weight', 1.0)
                for v in G.neighbors(u)
            )

        for i in range(len(nodes)):
            u = nodes[i]
            for j in range(i + 1, len(nodes)):
                v = nodes[j]
                
                correction = 0
                if G.has_edge(u, v):
                    w_uv = G[u][v].get('weight', 1.0)
                    correction = -2 * w_uv if x[u] == x[v] else 2 * w_uv
                    
                if gains[u] + gains[v] + correction > 0:
                    x[u] *= -1
                    x[v] *= -1
                    improved = True
                    break # Break inner loop to apply flip
            if improved:
                break # Break outer loop to completely recompute gains and restart
                
    return x

# =========================
# 5.2 Heuristic Wrappers for Registry
# =========================

def gw_single_rounding(G, V, **kwargs):
    num_samples = kwargs.get('num_samples', 100)
    x = _get_best_gw_partition(G, V, num_samples)
    x_opt = greedy_single(G, x)
    return _calculate_cut(G, x_opt)

def gw_pairwise_rounding(G, V, **kwargs):
    num_samples = kwargs.get('num_samples', 100)
    x = _get_best_gw_partition(G, V, num_samples)
    x_opt = greedy_pairwise(G, x)
    return _calculate_cut(G, x_opt)

def gw_single_then_pairwise_rounding(G, V, **kwargs):
    num_samples = kwargs.get('num_samples', 100)
    x = _get_best_gw_partition(G, V, num_samples)
    x1 = greedy_single(G, x)
    x_opt = greedy_pairwise(G, x1)
    return _calculate_cut(G, x_opt)

def gw_pairwise_then_single_rounding(G, V, **kwargs):
    num_samples = kwargs.get('num_samples', 100)
    x = _get_best_gw_partition(G, V, num_samples)
    x1 = greedy_pairwise(G, x)
    x_opt = greedy_single(G, x1)
    return _calculate_cut(G, x_opt)

# =========================
# 5.3 Sweepline Rounding (Parallel Shifting)
# =========================

def parallel_shift_rounding(G, V, **kwargs):
    """
    Optimized Goemans-Williamson with Parallel Shifting.
    Sweeps a threshold across the 1D projection to find the absolute optimal 
    hyperplane cut for each random vector instead of defaulting to T=0.
    """
    num_samples = kwargs.get('num_samples', 2000)
    n, d = V.shape
    
    # 1. Generate random hyperplanes and project
    R = np.random.randn(d, num_samples)
    R /= np.linalg.norm(R, axis=0)
    Y = V @ R  # Shape: (n, num_samples)
    
    best_overall_cut = 0
    
    # 2. Precompute a fast adjacency list using array indices for the inner loop
    nodes = list(G.nodes())
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    
    adj = [[] for _ in range(n)]
    for u, v, data in G.edges(data=True):
        w = data.get('weight', 1.0)
        idx_u, idx_v = node_to_idx[u], node_to_idx[v]
        adj[idx_u].append((idx_v, w))
        adj[idx_v].append((idx_u, w))

    # 3. Process each random projection
    for k in range(num_samples):
        y_proj = Y[:, k]
        
        # The O(N log N) Sort: Line up vertices from left to right along the 1D vector
        sorted_indices = np.argsort(y_proj)
        
        # Initialize: T is at -infinity. All nodes in Partition B.
        in_partition_A = np.zeros(n, dtype=bool)
        
        current_cut_value = 0
        max_cut_for_this_sample = 0
        
        # The O(E) Sweep
        for idx in sorted_indices:
            # Move node from Partition B to Partition A
            in_partition_A[idx] = True
            
            # Calculate local update
            edges_to_B = 0
            edges_to_A = 0
            for neighbor_idx, weight in adj[idx]:
                if in_partition_A[neighbor_idx]:
                    edges_to_A += weight
                else:
                    edges_to_B += weight
                    
            # The cut increases by edges broken (to B), decreases by edges rejoined (to A)
            current_cut_value += (edges_to_B - edges_to_A)
            
            # Track the best cut found along this specific 1D projection
            if current_cut_value > max_cut_for_this_sample:
                max_cut_for_this_sample = current_cut_value
                
        # Update the global best cut across all random samples
        if max_cut_for_this_sample > best_overall_cut:
            best_overall_cut = max_cut_for_this_sample
            
    return best_overall_cut




# --- UPDATED ROUNDING REGISTRY ---
ROUNDING_METHODS = {
    'gw': gw_rounding,
    'pca': pca_rounding,
    'gw_single': gw_single_rounding,
    'gw_pairwise': gw_pairwise_rounding,
    'gw_single_pairwise': gw_single_then_pairwise_rounding,
    'gw_pairwise_single': gw_pairwise_then_single_rounding,
    'parallel_shift': parallel_shift_rounding  
}

# =========================
# 6. Fallback: Fast Local Search
# =========================
def greedy_local_search(G, num_restarts=10):
    best_overall_cut = -np.inf
    nodes = list(G.nodes())

    for _ in range(num_restarts):
        partition = {node: np.random.choice([1, -1]) for node in nodes}
        improved = True
        
        while improved:
            improved = False
            for u in nodes:
                gain = sum(
                    G[u][v].get('weight', 1.0) if partition[u] == partition[v] else -G[u][v].get('weight', 1.0)
                    for v in G.neighbors(u)
                )
                if gain > 0:
                    partition[u] *= -1
                    improved = True

        current_cut = sum(data.get('weight', 1.0) for u, v, data in G.edges(data=True) if partition[u] != partition[v])
        if current_cut > best_overall_cut:
            best_overall_cut = current_cut

    return best_overall_cut

# =========================
# 7. Smart Pipeline Router
# =========================
def evaluate_maxcut(G, cvxpy_threshold=500, pymanopt_threshold=5000, rounding_config=None):
    """
    Evaluates MaxCut on G. Uses rounding_config to dynamically select post-SDP rounding method.
    """
    if rounding_config is None:
        rounding_config = {'method': 'gw', 'kwargs': {'num_samples': 2000}}
        
    rounding_name = rounding_config.get('method', 'gw')
    rounding_kwargs = rounding_config.get('kwargs', {})
    
    # Fetch the function from our registry, fallback to 'gw' if not found
    rounding_func = ROUNDING_METHODS.get(rounding_name, gw_rounding)

    n = G.number_of_nodes()
    t0 = time.perf_counter()

    if n <= cvxpy_threshold:
        print(f"  -> Routing to CVXPY (N={n} <= {cvxpy_threshold})")
        V, sdp_val = solve_sdp_cvxpy(G)
        cut = rounding_func(G, V, **rounding_kwargs)
        method = f"CVXPY + {rounding_name.upper()} Rounding"

    elif n <= pymanopt_threshold and PYMANOPT_AVAILABLE:
        print(f"  -> Routing to PyManopt Burer-Monteiro (N={n})")
        V, sdp_val = solve_burer_monteiro(G, rank=15)
        cut = rounding_func(G, V, **rounding_kwargs)
        method = f"PyManopt BM + {rounding_name.upper()} Rounding"

    else:
        print(f"  -> Routing to Greedy Heuristic (N={n} > {pymanopt_threshold})")
        cut = greedy_local_search(G, num_restarts=15)
        sdp_val = None
        method = "GREEDY_SEARCH"

    total_time = time.perf_counter() - t0
    approx_ratio = cut / sdp_val if sdp_val else None

    return {
        "graph_name": getattr(G, "name", "Unnamed"),
        "method": method,
        "nodes": n,
        "edges": G.number_of_edges(),
        "sdp_val": sdp_val,
        "cut_val": cut,
        "approx_ratio": approx_ratio,
        "total_time": total_time
    }

def build_experiment_configs():
    """
    Programmatically generates a compact suite of graph configurations
    across different sizes, densities, and topologies for RAPID testing.
    """
    configs = []

    # 1. Define Node Scales to test (Kept small for rapid iteration)
    node_scales = [30, 100]

    print(f"Building RAPID configurations for N scales: {node_scales}...")

    # ==========================================
    # 2. RANDOM GRAPHS (Representative densities)
    # ==========================================
    for n in node_scales:
        
        # Erdos-Renyi: Test Sparse and Medium (Avoided p=0.5 to prevent O(N^2) edge bloat)
        for p in [0.1, 0.3]:
            configs.append({'type': 'erdos_renyi', 'n': n, 'p': p})

        # Barabasi-Albert (Scale-free): Single representative attachment
        configs.append({'type': 'barabasi_albert', 'n': n, 'm': 3})

        # Watts-Strogatz (Small-world): Single representative
        configs.append({'type': 'watts_strogatz', 'n': n, 'k': 4, 'p': 0.1})

        # Random Regular: 
        if (n * 3) % 2 == 0: 
            configs.append({'type': 'regular', 'n': n, 'd': 3})

        # Powerlaw Cluster & Geometric
        configs.append({'type': 'powerlaw_cluster', 'n': n, 'm': 3, 'p': 0.1})
        configs.append({'type': 'geometric', 'n': n, 'radius': 0.25})

    # ==========================================
    # 3. CLASSIC GRAPHS
    # ==========================================
    for n in node_scales: 
        # Complete graphs generate N*(N-1)/2 edges. Cap it to avoid solver freeze.
        if n <= 50:
            configs.append({'type': 'complete', 'n': n})
            
        configs.append({'type': 'cycle', 'n': n})
        configs.append({'type': 'star', 'n': n})
        configs.append({'type': 'wheel', 'n': n})
        configs.append({'type': 'circular_ladder', 'n': n})
        configs.append({'type': 'random_tree', 'n': n})

        # Bipartite (Balanced vs. Imbalanced)
        configs.append({'type': 'complete_bipartite', 'n1': n//2, 'n2': n - (n//2)})
        if n >= 10:
            configs.append({'type': 'complete_bipartite', 'n1': int(n*0.2), 'n2': int(n*0.8)})

    # ==========================================
    # 4. LATTICES AND GRIDS
    # ==========================================
    # 2D Grids (N = 25, 100)
    for dim in [5, 10]:  
        configs.append({'type': 'grid_2d', 'dim1': dim, 'dim2': dim})

    # Hexagonal Lattices
    configs.append({'type': 'hexagonal', 'm': 3})
    configs.append({'type': 'hexagonal', 'm': 5})

    # Hypercubes (N = 16, 64)
    for dim in [4, 6]:    
        configs.append({'type': 'hypercube', 'dim': dim})

    # Balanced Trees (N = 31, 121)
    configs.append({'type': 'balanced_tree', 'r': 2, 'h': 4})
    configs.append({'type': 'balanced_tree', 'r': 3, 'h': 4})

    # ==========================================
    # 5. FIXED REAL-WORLD / SOCIAL NETWORKS
    # ==========================================
    configs.append({'type': 'karate_club'})
    configs.append({'type': 'florentine_families'})
    configs.append({'type': 'les_miserables'})

    # ==========================================
    # 6. STANFORD GSET BENCHMARKS (Just 1 for testing)
    # ==========================================
    # G1 has 800 nodes. Enough to test the PyManopt pipeline routing without taking forever.
    configs.append({'type': 'gset', 'id': 'G1'})

    print(f"Successfully generated {len(configs)} unique RAPID experiment configurations.")
    return configs


# =========================
# RUN BATCH EXPERIMENTS
# =========================
if __name__ == "__main__":
    experiment_configs = build_experiment_configs()
    
    print("Generating/Fetching graphs...")
    graphs = generate_graphs(experiment_configs)
    
    print("\nStarting Benchmark Pipeline...\n" + "="*65)
    
    # We will test two different rounding configurations for each graph
    rounding_tests = [
        {'method': 'gw', 'kwargs': {'num_samples': 2000}},
        {'method': 'gw_single', 'kwargs': {'num_samples': 100}},
        {'method': 'gw_single_pairwise', 'kwargs': {'num_samples': 100}},
        {'method': 'parallel_shift', 'kwargs': {'num_samples': 2000}}
    ]
    # Create a list to store all runs if you want to export to pandas/CSV later
    all_results = []
    
    for G in graphs:
        for r_conf in rounding_tests:
            print(f"Solving {G.name} with {r_conf['method'].upper()} rounding...")
            
            res = evaluate_maxcut(G, rounding_config=r_conf)
            all_results.append(res)
            
            print(f"  ├─ Method Used:     {res['method']}")
            if res['sdp_val']:
                print(f"  ├─ SDP Upper Bound: {res['sdp_val']:.2f}")
                print(f"  ├─ Approx Ratio:    {res['approx_ratio']:.4f}")
            else:
                print(f"  ├─ SDP Upper Bound: Skipped")
                
            print(f"  ├─ Actual Cut:      {res['cut_val']}")
            print(f"  └─ Total Time:      {res['total_time']:.3f}s\n")
        print("-" * 65)

Building configurations for N scales: [50, 501]...
Successfully generated 74 unique experiment configurations.
Generating/Fetching graphs...
Fetching G1 from Stanford repository...
Fetching G14 from Stanford repository...
Fetching G22 from Stanford repository...

Starting Benchmark Pipeline...
Solving ERDOS_RENYI_V50_E55_0 with GW rounding...
  -> Routing to CVXPY (N=50 <= 500)
  ├─ Method Used:     CVXPY + GW Rounding
  ├─ SDP Upper Bound: 52.84
  ├─ Approx Ratio:    0.9841
  ├─ Actual Cut:      52.0
  └─ Total Time:      0.117s

Solving ERDOS_RENYI_V50_E55_0 with GW_SINGLE rounding...
  -> Routing to CVXPY (N=50 <= 500)
  ├─ Method Used:     CVXPY + GW_SINGLE Rounding
  ├─ SDP Upper Bound: 52.84
  ├─ Approx Ratio:    0.9841
  ├─ Actual Cut:      52.0
  └─ Total Time:      0.134s

Solving ERDOS_RENYI_V50_E55_0 with GW_SINGLE_PAIRWISE rounding...
  -> Routing to CVXPY (N=50 <= 500)
  ├─ Method Used:     CVXPY + GW_SINGLE_PAIRWISE Rounding
  ├─ SDP Upper Bound: 52.84
  ├─ Approx Ratio: 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def visualize_benchmark_results(all_results):
    """
    Generates a 2x2 grid of visualizations comparing accuracy (approx ratio)
    and time taken across different solvers and rounding methods.
    """
    # 1. Convert the list of dicts to a pandas DataFrame
    df = pd.DataFrame(all_results)
    
    # Drop rows where approx_ratio is None (e.g., Greedy searches without SDP bounds) 
    # if you only want to compare bounded accuracy. Alternatively, keep them to see time.
    df_bounded = df.dropna(subset=['approx_ratio']).copy()

    # Set up the visualization grid
    sns.set_theme(style="whitegrid", palette="muted")
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    fig.suptitle('MaxCut Heuristics Benchmark Dashboard', fontsize=20, fontweight='bold', y=0.98)

    # -------------------------------------------------------------------------
    # PANEL 1: Trade-off - Time vs. Approximation Ratio
    # -------------------------------------------------------------------------
    sns.scatterplot(
        data=df_bounded, 
        x='total_time', 
        y='approx_ratio', 
        hue='method', 
        style='method',
        s=100, 
        alpha=0.7, 
        ax=axes[0, 0]
    )
    axes[0, 0].set_title('Performance Trade-off: Accuracy vs. Compute Time', fontsize=14)
    axes[0, 0].set_xscale('log')
    axes[0, 0].set_xlabel('Total Time (Seconds) [Log Scale]', fontsize=12)
    axes[0, 0].set_ylabel('Approximation Ratio (Cut / SDP Bound)', fontsize=12)
    axes[0, 0].legend(title='Method', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

    # -------------------------------------------------------------------------
    # PANEL 2: Distribution of Approximation Ratios
    # -------------------------------------------------------------------------
    sns.boxplot(
        data=df_bounded, 
        x='approx_ratio', 
        y='method', 
        hue='method',
        ax=axes[0, 1], 
        orient='h',
        legend=False
    )
    # Add a swarmplot on top to see individual data points
    sns.swarmplot(
        data=df_bounded, 
        x='approx_ratio', 
        y='method', 
        color=".2", 
        size=4, 
        alpha=0.6, 
        ax=axes[0, 1],
        orient='h'
    )
    axes[0, 1].set_title('Accuracy Distribution (Optimality Gap)', fontsize=14)
    axes[0, 1].set_xlabel('Approximation Ratio (Higher = Closer to Optimal)', fontsize=12)
    axes[0, 1].set_ylabel('')

    # -------------------------------------------------------------------------
    # PANEL 3: Scalability - Graph Size (Nodes) vs. Total Time
    # -------------------------------------------------------------------------
    sns.lineplot(
        data=df, 
        x='nodes', 
        y='total_time', 
        hue='method', 
        marker='o', 
        err_style="bars", 
        ax=axes[1, 0]
    )
    axes[1, 0].set_title('Scalability: Graph Size vs. Time', fontsize=14)
    axes[1, 0].set_xscale('log')
    axes[1, 0].set_yscale('log')
    axes[1, 0].set_xlabel('Number of Nodes (N) [Log Scale]', fontsize=12)
    axes[1, 0].set_ylabel('Total Time (Seconds) [Log Scale]', fontsize=12)
    axes[1, 0].legend(title='Method', fontsize=9)

    # -------------------------------------------------------------------------
    # PANEL 4: Distribution of Compute Times
    # -------------------------------------------------------------------------
    sns.boxplot(
        data=df, 
        x='total_time', 
        y='method', 
        hue='method',
        ax=axes[1, 1], 
        orient='h',
        legend=False
    )
    axes[1, 1].set_title('Compute Time Distribution by Method', fontsize=14)
    axes[1, 1].set_xscale('log')
    axes[1, 1].set_xlabel('Total Time (Seconds) [Log Scale]', fontsize=12)
    axes[1, 1].set_ylabel('')

    # Adjust layout to prevent overlap and show plot
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    # Save the plot (optional)
    plt.savefig('maxcut_benchmark_results.png', dpi=300, bbox_inches='tight')
    print("Saved benchmark visualization to 'maxcut_benchmark_results.png'")
    
    # Display the plot
    plt.show()

# --- To execute this, add the following to the very end of your script: ---
# if len(all_results) > 0:
#     print("\nGenerating benchmark visual dashboard...")
#     visualize_benchmark_results(all_results)